In [2]:
import pandas as pd

In [11]:
df_kalpi_address = pd.read_csv("../data/kalpi_address.csv")
columns = df_kalpi_address.columns
map_column_to_index = {
    "city_code": 2,
    "city_name": 3,
    "kalpi_code": 4,
    "kalpi_address": 6,
}
df_kalpi_address[[columns[map_column_to_index["kalpi_address"]], columns[map_column_to_index["city_name"]]]].head()

# df_kalpi_address["kalpi_full_address"] = (
#     df_kalpi_address[columns[map_column_to_index["kalpi_address"]]]
#     + ", "
#     + df_kalpi_address[columns[map_column_to_index["city_name"]]]
# )

# df_kalpi_address.to_csv("../data/kalpi_address.csv", index=False, encoding="utf-8")

,כתובת קלפי,שם ישוב בחירות
0,"מעגלי הרי""ם לוין,27",ירושלים
1,"שמואל הנביא,85",ירושלים
2,"מגן האלף,1",ירושלים
3,"פישל אהרן,29",ירושלים
4,"שבטי ישראל,27",ירושלים


In [10]:
import importlib
import sys
from pathlib import Path
sys.path.append('/workspaces/kartokalpi')
import research.geocoding
importlib.reload(research.geocoding)
from research.geocoding import GeocodingService

geo_cache_path = Path("./geocoding_cache.json")
geocoding_service = GeocodingService(geo_cache_path, enable_remote=True)

In [ ]:
import tqdm
for row in tqdm.tqdm(df_kalpi_address.itertuples(), total=len(df_kalpi_address)):
    locality_name = row[map_column_to_index["city_name"] + 1]
    kalpi_address = row[map_column_to_index["kalpi_address"] + 1]
    address = f"{kalpi_address},{locality_name}"
    coords = await geocoding_service.get_coordinates(f"{address}")
    # print(f"{address} -> {coords}")

100%|██████████| 11547/11547 [11:59<00:00, 16.04it/s] 


In [ ]:
import asyncio

async def geocode_all_addresses():
    addresses = [
        f"{row[map_column_to_index['kalpi_address'] + 1]},{row[map_column_to_index['city_name'] + 1]}"
        for row in df_kalpi_address.itertuples()
    ]
    
    # Run all geocoding tasks concurrently
    coords = await asyncio.gather(*[geocoding_service.get_coordinates(addr) for addr in addresses])
    return coords

# coords = await geocode_all_addresses()


In [57]:
none_count = coords.count(None)
print(f"Number of None values in coords: {none_count} out of {len(coords)}")


Number of None values in coords: 2588 out of 11547


In [59]:
df_kalpi_address["coordinates"] = coords
df_kalpi_address.to_csv("../data/kalpi_address_with_coords.csv", index=False, encoding="utf-8-sig")

In [4]:
df_kalpi_address = pd.read_csv("../data/kalpi_address_with_coords.csv")

In [ ]:
def make_full_address(row):
    locality_name = row["locality_name"]
    kalpi_address = row["kalpi_address"]
    full_address = f"{kalpi_address},{locality_name}"
    return full_address

async def geocode_missing_addresses(df_addresses):
    addresses = [
        make_full_address(row)
        for _, row in df_addresses[df_addresses.coordinates.isna()].iterrows()
    ]
    
    # Run all geocoding tasks concurrently
    coords = await asyncio.gather(*[geocoding_service.get_coordinates(addr) for addr in addresses])
    return coords

coords = await geocode_missing_addresses(df_kalpi_address)

HTTPStatusError: Client error '429 Too many requests' for url 'https://nominatim.openstreetmap.org/search?q=%D7%96%D7%A0%D7%92%D7%95%D7%99%D7%9C%2C29+%2C%D7%99%D7%A8%D7%95%D7%A9%D7%9C%D7%99%D7%9D%2C+%D7%99%D7%A9%D7%A8%D7%90%D7%9C&format=json&limit=1'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/429

/usr/local/lib/python3.13/asyncio/locks.py:200: RuntimeWarning: coroutine 'GeocodingService.get_coordinates' was never awaited
  async def wait(self):


In [42]:
# geocoding_service.get_coordinates
_full_address = make_full_address(df_kalpi_address[df_kalpi_address.coordinates.isna()].iloc[0])
display(_full_address)
await geocoding_service.get_coordinates(_full_address)

'פישל אהרן,29 ,ירושלים'

CancelledError: 

In [36]:
df_kalpi_address[df_kalpi_address.coordinates.isna()].iloc[0]

locality_id                     3000
locality_name                ירושלים
kalpi_id                           4
kalpi_address          פישל אהרן,29 
kalpi_location    ת"ת זיכרו תורת משה
coordinates                      NaN
Name: 3, dtype: object

In [43]:
df_kalpi_address


,locality_id,locality_name,kalpi_id,kalpi_address,kalpi_location,coordinates
0,3000,ירושלים,1,"מעגלי הרי""ם לוין,27","ביה""ס בית יעקב הצפון (סנהדריה)","(31.8033163, 35.2167565)"
1,3000,ירושלים,2,"שמואל הנביא,85","ת""ת שערי תורה (למען אחי)","(31.7944558, 35.2207062)"
2,3000,ירושלים,3,"מגן האלף,1",סמינר בית יעקב - עטרת חן,"(31.7948537, 35.2230044)"
3,3000,ירושלים,4,"פישל אהרן,29","ת""ת זיכרו תורת משה",NaN
4,3000,ירושלים,5,"שבטי ישראל,27",משרד החינוך,"(31.7841826, 35.2247057)"
...,...,...,...,...,...,...
11542,8800,שפרעם,39,אבו שהאב,"בי""ס מקיף ד - אבו שהאב","(32.807138, 35.189912)"
11543,8800,שפרעם,40,שפרעם,מרכז תרבות - מזרחי,"(32.8033274, 35.1959994)"
11544,8800,שפרעם,41,"מרכז מסחר-ע עתיקה,1","בי""ס קתולי חדש",NaN
11545,8800,שפרעם,42,שפרעם,"בי""ס יסודי מרשאן","(32.8033274, 35.1959994)"
